Dataset: a small real ratings dataset — MovieLens 100K (u.data) is the standard choice, small enough to run SGD on quickly, and it's the direct real-world analog of the Netflix case study in the chapter.





Load ratings into a sparse (user, movie, rating) format — build the mask Ω of known entries rather than a dense matrix.





Implement the SGD update from eq 9.9/9.10 directly (not full-batch gradient descent — the chapter's whole point in 9.3.2 is that SGD/sampling is what makes this tractable) — pick a small rank r (e.g. 5–10).





Train, tracking loss (RMSE) on a held-out test split of ratings.

In [3]:
import numpy as np
import random
import urllib.request
url = "https://files.grouplens.org/datasets/movielens/ml-100k/u.data"
urllib.request.urlretrieve(url, "u.data")
##build out sparse matrix (dataset only contains rows for ratings that actually exist, anything that would be blank in the matrix is omitted)
omega = {}
unique_users = set()
unique_movies = set()
with open("u.data", "r") as f:
  for line in f:
        fields = line.strip().split("\t")
        user_id, item_id, rating, timestamp = fields
        omega[(user_id, item_id)] = int(rating)
        unique_users.add(user_id)
        unique_movies.add(item_id)
num_users = len(unique_users)
num_movies = len(unique_movies)
user_id_to_index = {user_id:idx for idx, user_id in enumerate(unique_users)}
movie_id_to_index = {item_id:idx for idx, item_id in enumerate(unique_movies)}
#split omega into train/test data
omega_pairs = list(omega.keys())
random.shuffle(omega_pairs)
omega_cutoff = int(0.8*len(omega_pairs))
omega_train = omega_pairs[ :omega_cutoff]
omega_test = omega_pairs[omega_cutoff: ]
#set up factor matrices A (users × r) and B (movies × r) by randomly initializing A and B, and the iteratively update their values using gradient descent
#note: gd is based on the error between the current prediction A[user] dot B[movie] vs. the true rating, with row u being user u's latent vector, row i being movie i's latent vector
r = 5
eta = 0.01
A = np.random.rand(num_users, r)*0.1
B = np.random.rand(num_movies, r)*0.1
iterations = 50
while iterations != 0:
  error_sum_squared_train = 0
  error_count_train = 0
  RMSE_train = 0
  random.shuffle(omega_train)
  for (user_id, item_id) in omega_train:
    rating = omega[(user_id, item_id)]
    prediction = A[user_id_to_index[user_id]]@B[movie_id_to_index[item_id]]
    error = rating-prediction
    error_sum_squared_train += error**2
    error_count_train += 1
#gradient descent and RMSE comparison
    A_before_update = A[user_id_to_index[user_id]].copy()
    A[user_id_to_index[user_id]] = A[user_id_to_index[user_id]] + (eta*error*B[movie_id_to_index[item_id]])
    B[movie_id_to_index[item_id]] = B[movie_id_to_index[item_id]] + (eta*error*A_before_update)
  RMSE_train = np.sqrt(error_sum_squared_train/error_count_train)
  iterations -= 1
print(f"RMSE_train = {RMSE_train:.4f}")
error_sum_squared_test = 0
error_count_test = 0
RMSE_test = 0
for (user_id, item_id) in omega_test:
  rating = omega[(user_id, item_id)]
  prediction = A[user_id_to_index[user_id]]@B[movie_id_to_index[item_id]]
  error = rating-prediction
  error_sum_squared_test += error**2
  error_count_test += 1
RMSE_test = np.sqrt(error_sum_squared_test/error_count_test)
print(f"RMSE_test = {RMSE_test:.4f}")


RMSE_train = 0.8126
RMSE_test = 0.9471
